# Spark vs Flink – Kafka Streaming Benchmark Analysis

This notebook analyzes end-to-end behaviour of the **Kafka → Engine → Kafka** pipeline from the
`spark-flink-kafka-benchmark-fixed` project.

It is designed to:

- Load captured outputs from Kafka `topic2` for **Flink** and **Spark** runs
- Parse the line format used in this project
- Compute basic statistics:
  - record counts
  - simple proxy of throughput
  - latency metrics if timestamps `t0` / `t3` are available
- Visualize and compare Flink vs Spark in a reproducible way

You can commit this notebook to the repository (e.g. as `notebooks/benchmark_analysis.ipynb`)
and re-run it for different experiment runs.

## 1. Preparing data from Kafka

For each engine (Flink, Spark) we will:

1. Start the engine's job so that it writes to `topic2`.
2. Capture the output of `topic2` into a text file.
3. Analyze those text files here.

Example commands (run from the project root):

### 1.1 Capture Flink output

```bash
# In one terminal – start Flink job (as in README)
docker compose exec jobmanager bash -lc '
  cd /opt/flink/jobs/linearregression &&
  mvn -q -DskipTests package &&
  /opt/flink/bin/flink run -d -m jobmanager:8081 -p 2     -c app.LinearRegressionStreamProcessor     target/linearregression-1.0.1.jar
'

# In another terminal – capture topic2 to local file
docker compose exec kafka bash -lc   "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic topic2 --from-beginning"   > data/topic2_flink.txt
```

### 1.2 Capture Spark output (once Spark job is working)

```bash
# Start Spark job (example; see README for details)
docker compose exec jobmanager bash -lc '
  cd /opt/sparkjob && mvn -q -DskipTests package &&   /opt/spark/bin/spark-submit     --class app.LinearRegression     --master local[*]     --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.2     target/spark-linearregression-1.0.1.jar
'

# Capture topic2 output to a separate file
docker compose exec kafka bash -lc   "/opt/kafka/bin/kafka-console-consumer.sh --bootstrap-server kafka:9092 --topic topic2 --from-beginning"   > data/topic2_spark.txt
```

> You can repeat these steps for different experiment configurations
> (e.g. different `--count` in `dsta.py`) and keep multiple files in `data/`.

## 2. Notebook configuration

Set the paths to captured Kafka output files here.

In [ ]:
from pathlib import Path

# Adjust these paths to your local repository layout
DATA_DIR = Path("data")

FLINK_FILE = DATA_DIR / "topic2_flink.txt"
SPARK_FILE = DATA_DIR / "topic2_spark.txt"  # optional / when Spark is ready

print("Flink file:", FLINK_FILE.resolve())
print("Spark file:", SPARK_FILE.resolve())

## 3. Imports and helper functions

In [ ]:
import math
from pathlib import Path
from typing import List, Dict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

### 3.1 Parsing `topic2` line format

By default, the Flink job writes lines in the form:

```
f0,f1,...,fn;label;prediction;t1
```

If you extend the pipeline to include more timestamps (e.g. `t0` from the producer, `t3` at the consumer),
this parser is flexible enough to accept lines like:

```
f0,f1,...,fn;label;prediction;t1;t0;t3
```

and will create additional columns `t0`, `t3` when present.

In [ ]:
def parse_topic2_file(path: Path) -> pd.DataFrame:
    """Parse a topic2 dump file into a DataFrame.

    Expected base format:
        features_str;label;prediction;t1[;t0;t3]

    - features_str: comma-separated floats
    - label: float
    - prediction: float
    - t1: long/int (e.g. System.nanoTime at Flink/Spark stage)
    - t0, t3: optional timestamps (producer/consumer)

    The function is defensive: it skips empty lines and lines that cannot be parsed.
    """
    records = []

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    with path.open("r") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(";")
            if len(parts) < 4:
                continue

            features_str, label_str, pred_str, *rest = parts
            try:
                features = [float(x) for x in features_str.split(",") if x]
                label = float(label_str)
                prediction = float(pred_str)
            except ValueError:
                continue

            rec = {
                "features": features,
                "n_features": len(features),
                "label": label,
                "prediction": prediction,
            }

            if rest:
                try:
                    rec["t1"] = int(rest[0])
                except ValueError:
                    rec["t1"] = np.nan

            if len(rest) >= 2:
                try:
                    rec["t0"] = int(rest[1])
                except ValueError:
                    rec["t0"] = np.nan
            if len(rest) >= 3:
                try:
                    rec["t3"] = int(rest[2])
                except ValueError:
                    rec["t3"] = np.nan

            records.append(rec)

    df = pd.DataFrame.from_records(records)
    return df

## 4. Load and inspect Flink run

In [ ]:
flink_df = parse_topic2_file(FLINK_FILE)
flink_df.head()

In [ ]:
flink_df.describe(include="all")

### 4.1 Basic metrics for Flink run

In [ ]:
def summarize_run(df: pd.DataFrame, label: str) -> pd.DataFrame:
    summary = {}

    summary["engine"] = label
    summary["num_records"] = len(df)
    summary["n_features_mean"] = float(df["n_features"].mean()) if "n_features" in df else np.nan

    if {"t0", "t3"}.issubset(df.columns):
        valid = df.dropna(subset=["t0", "t3"]).copy()
        if not valid.empty:
            valid["latency"] = valid["t3"] - valid["t0"]
            summary["latency_count"] = int(len(valid))
            summary["latency_mean"] = float(valid["latency"].mean())
            summary["latency_p95"] = float(valid["latency"].quantile(0.95))
            summary["latency_p99"] = float(valid["latency"].quantile(0.99))
        else:
            summary["latency_count"] = 0
            summary["latency_mean"] = np.nan
            summary["latency_p95"] = np.nan
            summary["latency_p99"] = np.nan
    else:
        summary["latency_count"] = 0
        summary["latency_mean"] = np.nan
        summary["latency_p95"] = np.nan
        summary["latency_p99"] = np.nan

    return pd.DataFrame([summary])

flink_summary = summarize_run(flink_df, "flink")
flink_summary

### 4.2 (Optional) Flink latency distribution

In [ ]:
if {"t0", "t3"}.issubset(flink_df.columns):
    valid = flink_df.dropna(subset=["t0", "t3"]).copy()
    valid["latency"] = valid["latency"] = valid["t3"] - valid["t0"]

    plt.figure()
    plt.hist(valid["latency"], bins=50)
    plt.xlabel("Latency (t3 - t0)")
    plt.ylabel("Count")
    plt.title("Flink end-to-end latency distribution")
    plt.show()
else:
    print("No t0/t3 columns found in Flink data; latency histogram skipped.")

## 5. Load and inspect Spark run (optional)

In [ ]:
if SPARK_FILE.exists():
    spark_df = parse_topic2_file(SPARK_FILE)
    display(spark_df.head())
    spark_summary = summarize_run(spark_df, "spark")
    display(spark_summary)
else:
    print("Spark file not found; skipping Spark analysis.")

## 6. Flink vs Spark comparison

In [ ]:
summaries = []
if "flink_summary" in globals():
    summaries.append(flink_summary)
if "spark_summary" in globals():
    summaries.append(spark_summary)

if summaries:
    all_summary = pd.concat(summaries, ignore_index=True)
    display(all_summary)

    if all_summary["latency_count"].max() > 0:
        metrics_to_plot = ["latency_mean", "latency_p95", "latency_p99"]
        plot_df = all_summary.set_index("engine")[metrics_to_plot]

        ax = plot_df.plot(kind="bar")
        ax.set_ylabel("Latency (time units of t3 - t0)")
        ax.set_title("Flink vs Spark latency comparison")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        plt.tight_layout()
        plt.show()
    else:
        print("No latency metrics available (no t0/t3). Only counts and feature stats are shown above.")
else:
    print("No summaries to compare yet.")

## 7. Next steps

To turn this into a proper benchmark for your thesis:

- Extend the data format to propagate `t0` (producer timestamp) and `t3` (measurement at consumer or log time),
  so that end-to-end latency can be measured precisely.
- Run several experiments for different loads (e.g. `--count` = 10K, 100K, 1M, 5M).
- Store each run's `topic2_*.txt` in `data/runs/...` with clear naming.
- Re-run this notebook and export results/plots as figures for the thesis.

This notebook is intentionally simple, so you can easily adapt it:
- add CPU/RAM stats from Docker or external monitoring,
- add per-partition analysis,
- or additional metrics specific to your use case.